# Verify cameras

This notebook inventories connected cameras, tests frame capture, and shows labelled live previews so that each physical camera can be matched to its terminal. Press **q** in a preview window to exit it.

In [2]:
import sys
import re
from pathlib import Path
from pprint import pprint

project_root = next((path for path in (Path.cwd(), *Path.cwd().parents)
                     if (path / 'src' / 'exp_run_config.py').is_file()), None)
if project_root is None:
    raise RuntimeError('Run this notebook from the BerryPicker repository or one of its subdirectories.')
sys.path.insert(0, str(project_root / 'src'))

import cv2
from exp_run_config import Config
Config.PROJECTNAME = 'BerryPicker'
print(f'OpenCV version: {cv2.__version__}')

OpenCV version: 5.0.0


In [3]:
def video_index(device_path):
    match = re.search(r'video(\d+)$', str(device_path))
    return int(match.group(1)) if match else None

# Stable names identify the USB terminal more reliably than probing arbitrary indices.
stable_paths = []
for directory in (Path('/dev/v4l/by-id'), Path('/dev/v4l/by-path')):
    if directory.exists():
        stable_paths.extend(sorted(directory.glob('*-video-index*')))

inventory = []
seen_device_paths = set()
for stable_path in stable_paths:
    device_path = stable_path.resolve()
    if device_path in seen_device_paths:
        continue
    seen_device_paths.add(device_path)
    cap = cv2.VideoCapture(str(stable_path), cv2.CAP_V4L2)
    try:
        opened = cap.isOpened()
        read_ok, frame = cap.read() if opened else (False, None)
        inventory.append({
            'opencv_id': video_index(device_path),
            'stable_path': str(stable_path),
            'device_path': str(device_path),
            'opened': opened,
            'read_ok': read_ok,
            'frame_shape': tuple(frame.shape) if read_ok else None,
            'backend': cap.getBackendName() if opened else None,
        })
    finally:
        cap.release()

if inventory:
    for entry in inventory:
        status = 'PASS' if entry['read_ok'] else 'FAIL'
        print(f"{status}  dev{entry['opencv_id']}: {entry['stable_path']} -> {entry['device_path']}  frame={entry['frame_shape']}")
else:
    print('FAIL  No stable paths found under /dev/v4l/by-id or /dev/v4l/by-path.')

PASS  dev0: /dev/v4l/by-id/usb-Microsoft_Microsoft®_LifeCam_Cinema_TM_-video-index0 -> /dev/video0  frame=(480, 640, 3)
FAIL  dev1: /dev/v4l/by-id/usb-Microsoft_Microsoft®_LifeCam_Cinema_TM_-video-index1 -> /dev/video1  frame=None


[ WARN:0@7.761] global cap.cpp:212 open VIDEOIO(V4L2): backend is generally available but can't be used to capture by name


In [4]:
# Set this to the camera-controller configuration used by the intended setup.
camera_experiment = 'controllers'
camera_run = 'camera_controller_tredy2'
exp_camera = Config().get_experiment(camera_experiment, camera_run)
configured_ids = list(exp_camera['active_camera_list'])
working_ids = {entry['opencv_id'] for entry in inventory if entry['read_ok']}
print(f'Configured cameras: {configured_ids}')
for camera_id in configured_ids:
    print(f"{'PASS' if camera_id in working_ids else 'FAIL'}  configured dev{camera_id}")

preview_ids = configured_ids  # Change to any passing OpenCV device IDs if needed.

***ExpRun**: Loading pointer config file:
	/home/lboloni/.config/BerryPicker/mainsettings.yaml
***ExpRun**: Loading machine-specific config file:
	~/WORK/BerryPicker/cfg/settings.yaml
***ExpRun**: Using torch device: cuda
***ExpRun**: Experiment default config /home/lboloni/WORK/BerryPicker/src/BerryPicker/src/experiment_configs/controllers/_defaults_controllers.yaml was empty, ok.
***ExpRun**: Configuration for exp/run: controllers/camera_controller_tredy2 successfully loaded
Configured cameras: [0, 2]
PASS  configured dev0
FAIL  configured dev2


In [5]:
# Live labelled preview. Cover or move one physical camera at a time to verify its terminal mapping.
captures = {}
try:
    for camera_id in preview_ids:
        cap = cv2.VideoCapture(camera_id, cv2.CAP_V4L2)
        if cap.isOpened():
            captures[camera_id] = cap
        else:
            print(f'FAIL  Unable to open dev{camera_id}')
            cap.release()
    if not captures:
        raise RuntimeError('No configured cameras could be opened.')

    while True:
        tiles = []
        for camera_id, cap in captures.items():
            ok, frame = cap.read()
            if not ok:
                print(f'FAIL  dev{camera_id} stopped returning frames')
                continue
            frame = cv2.resize(frame, (320, 240))
            cv2.putText(frame, f'dev{camera_id}', (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
            tiles.append(frame)
        if not tiles:
            break
        cv2.imshow('Verify_Cameras — q to exit', cv2.hconcat(tiles))
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
finally:
    for cap in captures.values():
        cap.release()
    cv2.destroyAllWindows()
    print('Camera captures released.')

FAIL  Unable to open dev2


[ WARN:0@22.656] global cap_v4l.cpp:914 open VIDEOIO(V4L2:/dev/video2): can't open camera by index
[ WARN:0@22.656] global cap.cpp:475 open VIDEOIO(V4L2): backend is generally available but can't be used to capture by index
QFontDatabase: Cannot find font directory /home/lboloni/WORK/BerryPicker/vm/berrypickervenv/lib/python3.12/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/lboloni/WORK/BerryPicker/vm/berrypickervenv/lib/python3.12/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/lboloni/WORK/BerryPicker/vm/berrypickervenv/lib/python3.12/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.

Camera captures released.


A passing setup has a successful one-frame read for every configured device and a live feed whose physical view matches the camera expected at that terminal. Record or update the camera configuration only after completing that visual check.